In [0]:
# source
ECOMM_BRONZE_FILES = "abfss://bronze@gksdatalake4.dfs.core.windows.net/ecomm/"
# target
ECOMM_SILVER_FILES = "abfss://silver@gksdatalake4.dfs.core.windows.net/ecomm/"
# checkpoint, stores the state, for restart, or spark failover
CHECKPOINT_LOCATION = "abfss://bronze@gksdatalake4.dfs.core.windows.net/checkpoints/ecom-stream-bronze-to-silver"

In [0]:
# read from gksdatalake2, container bronze, ecomm from adls, csv files
# testing purpose only, not used
# BATCH SPARK.READ
df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(ECOMM_BRONZE_FILES)

df.printSchema()
df.show(2)

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType


# Step 1: Define explicit schema
ecomm_schema = StructType([
    StructField("InvoiceNo", StringType(), True),
    StructField("StockCode", StringType(), True),
    StructField("Description", StringType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("InvoiceDate", TimestampType(), True),
    StructField("UnitPrice", DoubleType(), True),
    StructField("CustomerID", IntegerType(), True),
    StructField("Country", StringType(), True)
])

# LAZY , not yet started the stream
# Step 1: Read from Bronze Zone using Auto Loader
bronze_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .schema(ecomm_schema)
    .load(ECOMM_BRONZE_FILES)  
)

# .show will not work on stream
bronze_df.printSchema()

In [0]:
# since show will not work, we can use display
# display will work,
bronze_df.display() # display(bronze_df)

In [0]:
import pyspark.sql.functions as F

# Step 2: Clean and filter data
# LAZY , not yet started the stream

cleaned_df = (
    bronze_df
    # Parse InvoiceDate from ISO format
    .withColumn("InvoiceDate", F.to_timestamp( F.col("InvoiceDate"), "yyyy-MM-dd HH:mm:ss"))

    # Ensure StockCode remains string
    .withColumn("StockCode",  F.col("StockCode").cast("string"))

    # Filter: remove rows where InvoiceNo starts with 'C' (returns)
    .filter(~ F.col("InvoiceNo").cast("string").startswith("C"))

    # Filter: remove rows with null CustomerID
    .filter( F.col("CustomerID").isNotNull())

    # Filter: remove rows with negative quantity
    .filter( F.col("Quantity") >= 0)

    # Partitioning columns
    .withColumn("year",  F.year("InvoiceDate"))
    .withColumn("month",  F.month("InvoiceDate"))
    .withColumn("day",  F.dayofmonth("InvoiceDate"))
)


cleaned_df.printSchema()

In [0]:
# LAZY , not yet started the stream

withAmountDf = cleaned_df.withColumn("Amount", F.col("Quantity") * F.col("UnitPrice"))
withAmountDf.printSchema()

In [0]:
# Step 3: Write to Silver zone
# STREAM ACTION
query = (
    withAmountDf.writeStream
    .format("delta")
    .outputMode("append")
    .partitionBy("year", "month", "day")
    .option("checkpointLocation", CHECKPOINT_LOCATION)
    .trigger(processingTime="1 minutes")
    .start(ECOMM_SILVER_FILES)
)

# query.stop() # to stop the query execution